# 七零大院：冷面糙汉宠妻成瘾

**กด ▶ ทีละ cell หรือ Runtime → Run all**

ไฟล์จะโหลดอัตโนมัติเมื่อเสร็จ (~10-15 นาที)

In [ ]:
!pip install -q requests beautifulsoup4

In [ ]:
import re, time, requests
from bs4 import BeautifulSoup
from pathlib import Path
from google.colab import files

BOOK_URL = 'https://ixdzs8.com/read/609163/'
OUTPUT   = '七零大院_冷面糙汉宠妻成瘾.txt'
DELAY    = 1.2

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/124.0.0.0 Safari/537.36',
    'Accept-Language': 'zh-CN,zh;q=0.9',
    'Referer': 'https://ixdzs8.com/',
}

def get_chapters():
    r = requests.get(BOOK_URL, headers=HEADERS, timeout=15)
    r.encoding = 'utf-8'
    soup = BeautifulSoup(r.text, 'html.parser')
    links = (soup.select('.clist a') or soup.select('.chapter-list a')
             or soup.select('.mulu a') or soup.select('ul.list a'))
    base = 'https://ixdzs8.com'
    return [(a.get_text(strip=True),
             a['href'] if a['href'].startswith('http') else base + a['href'])
            for a in links if a.get('href') and a.get_text(strip=True)]

def probe_chapters(max_ch=600):
    base = BOOK_URL.rstrip('/')
    chapters = []
    for n in range(1, max_ch + 1):
        url = f'{base}/p{n}.html'
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 404:
            print(f'Stopped at p{n}'); break
        r.encoding = 'utf-8'
        soup = BeautifulSoup(r.text, 'html.parser')
        t = soup.select_one('h1,.chapter-title,.title')
        chapters.append((t.get_text(strip=True) if t else f'第{n}章', url))
        if n % 50 == 0: print(f'  probed {n}…')
        time.sleep(DELAY)
    return chapters

def scrape(url):
    r = requests.get(url, headers=HEADERS, timeout=15)
    r.encoding = 'utf-8'
    soup = BeautifulSoup(r.text, 'html.parser')
    el = (soup.select_one('#page') or soup.select_one('.read-content')
          or soup.select_one('.content') or soup.select_one('article'))
    if not el: return ''
    for tag in el.select('script,style,.ad,ins'): tag.decompose()
    lines = [p.get_text(' ', strip=True) for p in el.find_all(['p','br'])]
    if not lines:
        lines = [l.strip() for l in el.get_text('\n').splitlines() if l.strip()]
    skip = re.compile(r'(ixdzs8|请记住本书首发域名|www\.|m\.ixdzs8|本章未完)', re.I)
    return '\n'.join(l for l in lines if not skip.search(l))

# --- main ---
print('กำลังดึง chapter list…')
chapters = get_chapters()
if not chapters:
    print('ไม่พบ chapter list ในหน้า — กำลัง probe p1, p2, …')
    chapters = probe_chapters()

print(f'พบ {len(chapters)} บท — กำลังดาวน์โหลด…\n')
with open(OUTPUT, 'w', encoding='utf-8') as f:
    for i, (title, url) in enumerate(chapters, 1):
        content = scrape(url)
        f.write(f'\n\n第{i}章　{title}\n\n{content}\n')
        if i % 20 == 0 or i == len(chapters):
            print(f'  [{i}/{len(chapters)}] {title}')
        time.sleep(DELAY)

print(f'\n✓ เสร็จแล้ว! กำลังโหลดไฟล์…')
files.download(OUTPUT)